# 外包数据清洗全流程：从原始交易数据到 `full_data` + 描述统计

本 notebook 从**最原始的年度清洗交易数据**开始，一步步重现到第一阶段实证所用的底表 `full_data.dta`，并给出核心描述统计。整合自 VM 上的 `database.do` / `to_nine` / `product_character` / `product_level` / `coverage` 等脚本，**全程采用统一的正确口径**。

## 数据血统

```
raw  1718_total_cleaned_by_year1.dta        (最原始，已按年清洗的交易数据)
 │ [Step1] collapse 到 firm×product×io×year，截 15 位
 ▼
 (lenth15)
 │ [Step2] 15 位码标准化为 9 位（4058 → 2778 产品），保留既产出又投入的企业
 ▼
 (lenth9)
 │ [Step3] firm×product×year 聚合；外包额 = min(投入, 产出)
 ▼
 firm_product_year_level.dta
 │ [Step4] firm×year 汇总：外包强度、中介/外包标记
 │ [Step5] 主产品 = 自产产值 production_value 最大
 │ [Step6] 合并 similarity
 ▼
 full_data.dta   ← 第一阶段回归底表
 │ [Step7] 描述统计
 ▼
 企业分类 / scope gap / 外包普遍率与强度
```

## 关键口径（本 notebook 统一采用）

- **外包产品**：同一企业同一年对同一产品既买(投入)又卖(产出)。
- **外包额** = `min(投入额, 产出额)`（逐 firm×product×year）——只算"买来又卖掉"的部分，买来自用的归自产。
- **自产额** `production_value` = 产出额 − 外包额。
- **外包强度** = `Σ外包额 / Σ产出额`（firm×year）。
- **中介** `is_intermediary` = 强度 > 0.90；**外包企业** `is_outsourcing` = 强度 ≥ 0.01。
- **主产品** = firm×year 内 `production_value` 最大者（并列取 `product_id` 最小，确定性）。

> ⚠️ **大文件**：最原始文件极大，Step1 的 collapse 交给 **Stata**（`database.do`）做；Step3 起在约 4.65 亿行的 lenth9 上用 pandas 操作，建议在 VM（大内存）上运行。
>
> ⚠️ **列名假设**：原始 `1718_total_cleaned_by_year1.dta` 含列 `firm_id, product_id, input_output('input'/'output'), v, year`。若不同请在 Step1 调整。


In [ ]:
import pandas as pd
import numpy as np
import gc, os
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# ==== 路径约定：代码 git 共享，生成数据只在 VM ====
CODE = Path(r'G:\Kuangyu_Temp\Outsource\Empirical1')     # 代码（git 同步，两边共享）
DATA = Path(r'G:\Kuangyu_Temp\Outsource\replicate')      # 所有生成数据（只在 VM，不进 git）
SRC  = Path(r'G:\Kuangyu_Temp\Outsource')                # 已有输入数据（similarity 等）
RAW  = Path(r'G:\Kuangyu_Temp\single_product\1718_total_cleaned_by_year1.dta')  # 最原始交易数据

SIM  = SRC / 'full_product_similarity.dta'    # 产品对相似度（输入，保持原位）

# 输出（全部落到 DATA）
OUT_FPY      = DATA / 'firm_product_year_level.dta'
OUT_FULLDATA = DATA / 'full_data.dta'

DATA.mkdir(exist_ok=True)
os.chdir(DATA)
print('CODE (代码, git):', CODE)
print('DATA (生成数据) :', DATA)
print('SRC  (已有输入) :', SRC)

## Step 1　原始交易 → lenth15（用 Stata `database.do`）

原始文件极大（可达数十亿行），**pandas / VS Code 打不开**，所以 collapse 交给 **Stata** 做（VM 能开、精确、无分块问题）。本格通过 `StataMP-64 /e do` 调用 `database.do`：

- `collapse (sum) v, by(firm_id product_id input_output year)`、drop `v<=0`、截 15 位、`is_output`、只保留有产出的企业 → 存 `lenth15.dta`。

**改 `DO_DATABASE` 为 VM 上 `database.do` 的实际路径。**

In [ ]:
import subprocess

stata_exe   = r"C:/Program Files/Stata17/StataMP-64.exe"
DO_DATABASE = str(CODE / 'replicate' / 'database.do')   # 代码在 Empirical1 里

print('running database.do in Stata ...')
proc = subprocess.run([stata_exe, "/e", "do", DO_DATABASE],
                      capture_output=True, text=True, errors="ignore")
print('Stata return code:', proc.returncode, '(0 = 正常)')

df15 = pd.read_stata(DATA / 'lenth15.dta')   # Stata 产出，落在 DATA
print('lenth15 行数:', f'{len(df15):,}',
      '| 企业:', df15['firm_id'].nunique(),
      '| 15位产品:', df15['product_id'].nunique())

## Step 2　15 位码 → 9 位码标准化（重写 `to_nine`）

中国的商品编码是**分层**的：位数越多越细（1/3/5/7/9 位为层级节点，后面补 0）。目标是把所有码统一到一致的 9 位层级，但**有些产品其实只细到 5 位或 7 位**（9 位只是补零）。逻辑：

1. **剔除高层聚合码**：1 位或 3 位后面全是 0 的（太粗）。
2. **判断真实细分层级**：若某 7 位前缀只对应一个 9 位码，说明该产品最多到 7 位；5 位同理。
3. **构造最终 9 位码集合** = 真正的 9 位码（不以 `00` 结尾）+ 7 位层级码（补 `00` 到 9 位）。
4. 用该集合过滤，并在 9 位层级**重新聚合**求和。
5. **只保留既在产出侧、又在投入侧出现的企业**。

结果：产品 4058 → **2778**，企业约 **7,191,877**。

In [ ]:
df1 = df15
df1['product_id'] = df1['product_id'].astype(str)
df2 = df1[df1['is_output'] == 1].copy()   # 产出侧
df3 = df1[df1['is_output'] == 0].copy()   # 投入侧
df2['product_id_9'] = df2['product_id'].str[:9]
df3['product_id_9'] = df3['product_id'].str[:9]

def is_high_level(code):
    # 1 位或 3 位聚合码（其余全为 0），过于粗，剔除
    return (code[1:] == '0' * (len(code) - 1)) or (code[3:] == '0' * (len(code) - 3))

df_low = df2[~df2['product_id'].apply(is_high_level)]

# 在低层码里判断每个产品真实的细分层级
g = df_low.groupby('product_id', as_index=False)['v'].sum()
g['p5'] = g['product_id'].str[:5]
g['p7'] = g['product_id'].str[:7]
g['p9'] = g['product_id'].str[:9]

c7 = g.groupby('p7')['p9'].nunique(); single7 = set(c7[c7 == 1].index)  # 7 位前缀唯一
c5 = g.groupby('p5')['p7'].nunique(); single5 = set(c5[c5 == 1].index)  # 5 位前缀唯一

five_in_seven  = [x for x in single7 if x.endswith('00')]      # 实为 5 位层级
seven_in_seven = [x for x in single7 if not x.endswith('00')]  # 7 位层级
for i in five_in_seven:
    if i[:-2] in single5:
        seven_in_seven.append(i)
seven_in_seven = [i + '00' for i in seven_in_seven]

p9_true = [x for x in g['p9'].drop_duplicates().tolist() if not x.endswith('00')]  # 真 9 位
product_id_9_final = set(p9_true + seven_in_seven)
print('最终合法 9 位码数:', len(product_id_9_final))

def to9(dd):
    dd = dd[dd['product_id_9'].isin(product_id_9_final)].copy()
    dd = dd.drop(columns=['product_id']).rename(columns={'product_id_9': 'product_id'})
    return dd.groupby(['firm_id', 'product_id', 'is_output', 'year'], as_index=False)['v'].sum()

out9 = to9(df2)
in9  = to9(df3)

firms = set(out9['firm_id']) & set(in9['firm_id'])   # 既产出又投入的企业
out9 = out9[out9['firm_id'].isin(firms)]
in9  = in9[in9['firm_id'].isin(firms)]

df9 = pd.concat([in9, out9], ignore_index=True)   # = lenth9
print('lenth9 行数:', f'{len(df9):,}',
      '| 企业:', df9['firm_id'].nunique(),
      '| 产品:', df9['product_id'].nunique())
del df1, df2, df3, df_low, g, out9, in9, df15; gc.collect()

## Step 3　firm×product×year 聚合，外包额 = min(投入, 产出)

以**产出侧为主表** left-merge 投入侧（因此只保留有产出/销售记录的 firm×product×year）。核心口径：

- `outsourcing_value = min(total_input, total_output)`（只算买来又卖掉的部分）
- `production_value  = total_output − outsourcing_value`（自产）
- `outsourcing_percen = outsourcing_value / total_output`

In [ ]:
out = (df9[df9['is_output'] == 1].groupby(['year', 'firm_id', 'product_id'], as_index=False)['v']
         .sum().rename(columns={'v': 'total_output'}))
inp = (df9[df9['is_output'] == 0].groupby(['year', 'firm_id', 'product_id'], as_index=False)['v']
         .sum().rename(columns={'v': 'total_input'}))

fpy = out.merge(inp, on=['year', 'firm_id', 'product_id'], how='left')
fpy['total_input'] = fpy['total_input'].fillna(0)

fpy['outsourcing_value']  = fpy[['total_input', 'total_output']].min(axis=1)
fpy['production_value']   = fpy['total_output'] - fpy['outsourcing_value']
fpy['outsourcing_percen'] = (fpy['outsourcing_value'] / fpy['total_output']).fillna(0)

fpy[['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value',
     'production_value', 'outsourcing_percen']].to_stata(OUT_FPY, write_index=False)
print('firm_product_year_level 行数:', f'{len(fpy):,}',
      '| firm-year:', fpy.groupby(['firm_id', 'year']).ngroups)
del df9, out, inp; gc.collect()

## Step 4　firm×year 汇总：外包强度、中介/外包标记

- `outsourcing_intensity = firm_total_outsource / firm_total_output`（min 口径）
- `is_intermediary = 强度 > 0.90`，`is_outsourcing = 强度 ≥ 0.01`

对照：新口径下中介占比约 **6.24%**（旧的"整块购入"口径为 12.46%）。

In [ ]:
fpy['year'] = fpy['year'].astype(int)
firm_summary = fpy.groupby(['year', 'firm_id'], as_index=False).agg(
    firm_total_output    = ('total_output',      'sum'),
    firm_total_outsource = ('outsourcing_value', 'sum'),
    n_products           = ('product_id',        'count'),
)
firm_summary['outsourcing_intensity'] = (
    firm_summary['firm_total_outsource'] / firm_summary['firm_total_output']).fillna(0)
firm_summary['is_intermediary'] = (firm_summary['outsourcing_intensity'] > 0.90).astype(int)
firm_summary['is_outsourcing']  = (firm_summary['outsourcing_intensity'] >= 0.01).astype(int)

print('firm-year 观测:', f"{len(firm_summary):,}")
print('中介占比:     {:.2%}'.format(firm_summary['is_intermediary'].mean()))
print('外包企业占比: {:.2%}'.format(firm_summary['is_outsourcing'].mean()))

## Step 5　主产品 = 自产产值 `production_value` 最大

按你确认的口径，主产品取 firm×year 内 **自产产值最大**者（并列取 `product_id` 最小，保证确定性），而不是总产出最大。同时构造：

- `sales_percen        = total_output / firm_total_output`
- `sales_relative_main = total_output / 主产品的 total_output`

In [ ]:
fpy_sorted = fpy.sort_values(['year', 'firm_id', 'production_value', 'product_id'],
                             ascending=[True, True, False, True])
main = (fpy_sorted.groupby(['year', 'firm_id'], as_index=False)
        .first()[['year', 'firm_id', 'product_id', 'total_output']]
        .rename(columns={'product_id': 'main_product', 'total_output': 'main_product_output'}))

df = fpy.merge(main, on=['year', 'firm_id'], how='left')
df['is_main'] = (df['product_id'] == df['main_product']).astype(int)

df = df.merge(firm_summary, on=['year', 'firm_id'], how='left')
df['sales_percen']        = df['total_output'] / df['firm_total_output']
df['sales_relative_main'] = df['total_output'] / df['main_product_output']
del fpy_sorted, main; gc.collect()
print('主产品并入完成，df 行数:', f'{len(df):,}')

## Step 6　合并 similarity → `full_data.dta`

`full_product_similarity.dta` 是产品对（product_1, product_2）的 `input_similarity` / `output_similarity`（来自投入产出表）。对称化后按 (product_id, main_product) 合并；**主产品与自身的相似度定义为 1**。

In [ ]:
sim = pd.read_stata(SIM)
sim1 = sim.rename(columns={'product_1': 'product_id', 'product_2': 'main_product'})
sim2 = sim.rename(columns={'product_2': 'product_id', 'product_1': 'main_product'})
sim_lu = (pd.concat([sim1, sim2], ignore_index=True)
            .drop_duplicates(subset=['product_id', 'main_product']))

for c in ['product_id', 'main_product']:
    df[c]     = df[c].astype(str).str.strip()
    sim_lu[c] = sim_lu[c].astype(str).str.strip()

df = df.merge(sim_lu[['product_id', 'main_product', 'input_similarity', 'output_similarity']],
              on=['product_id', 'main_product'], how='left')
df.loc[df['is_main'] == 1, ['input_similarity', 'output_similarity']] = 1.0

cols = ['year', 'firm_id', 'product_id', 'total_output', 'outsourcing_value', 'production_value',
        'outsourcing_percen', 'sales_percen', 'sales_relative_main', 'is_main', 'main_product',
        'main_product_output', 'input_similarity', 'output_similarity', 'firm_total_output',
        'firm_total_outsource', 'n_products', 'outsourcing_intensity', 'is_intermediary', 'is_outsourcing']
df = df[cols].sort_values(['year', 'firm_id', 'total_output'], ascending=[True, True, False]).reset_index(drop=True)
df.to_stata(OUT_FULLDATA, write_index=False)
print('full_data.dta 保存完成:', f'{len(df):,}', '行 |', df['firm_id'].nunique(), '企业 |', df['product_id'].nunique(), '产品')

## Step 7　描述统计

以下用新口径复现第一阶段的核心描述性事实。

### 7.1　企业分类（firm-year）

按外包强度把 firm-year 分成：中介（>0.90）/ 外包（0.01–0.90）/ 纯自产（<0.01）。

In [ ]:
fy = firm_summary.copy()
def classify(r):
    if r['is_intermediary'] == 1:            return '中介 Intermediary'
    if r['outsourcing_intensity'] >= 0.01:   return '外包 Outsourcing'
    return '纯自产 Pure self'
fy['ftype'] = fy.apply(classify, axis=1)

tab = fy.groupby('ftype').agg(
    firm_years   = ('firm_id', 'size'),
    unique_firms = ('firm_id', 'nunique'),
    total_output = ('firm_total_output', 'sum'),
).reset_index()
tab['pct_firm_years'] = tab['firm_years'] / tab['firm_years'].sum()
print(tab.to_string(index=False))

### 7.2　Scope gap：产品对的自产/混合/外包分解（剔除中介）

按 `outsourcing_percen` 把每个 firm×product×year 分成：纯自产(=0) / 混合(0–1) / 纯外包(=1)，看**占产品对比例**与**占销售额比例**。预期：混合型占对数约 20%，却贡献约 72% 销售额。

In [ ]:
non_int = df[df['is_intermediary'] != 1]
def bucket(p):
    if p <= 0: return '纯自产 Pure self'
    if p >= 1: return '纯外包 Pure outsourcing'
    return '混合 Mixed'
bk = non_int['outsourcing_percen'].apply(bucket)

sg = non_int.groupby(bk).agg(n_pairs=('total_output', 'size'),
                             sales=('total_output', 'sum'))
sg['pct_pairs'] = sg['n_pairs'] / sg['n_pairs'].sum()
sg['pct_sales'] = sg['sales']   / sg['sales'].sum()
print(sg[['n_pairs', 'pct_pairs', 'pct_sales']].to_string())

### 7.3　外包普遍率与强度分布（剔除中介）

In [ ]:
print('外包普遍率（is_outsourcing，按年）:')
print(firm_summary.groupby('year')['is_outsourcing'].mean().round(4).to_string())

os_firms = firm_summary[(firm_summary['is_intermediary'] == 0) &
                        (firm_summary['outsourcing_intensity'] > 0)]
print('\n外包强度分布（剔除中介、强度>0）:')
print(os_firms['outsourcing_intensity'].describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(4).to_string())

## 输出文件

| 文件 | 内容 |
|---|---|
| `firm_product_year_level.dta` | firm×product×year：total_output / outsourcing_value / production_value / outsourcing_percen |
| `full_data.dta` | 第一阶段回归底表（含主产品、similarity、firm 汇总、is_intermediary 等 20 列）|

**口径备注**：外包额 = min(投入,产出)；主产品 = production_value 最大；中介 = 强度>0.90。与旧版差异见 `DATA_LINEAGE`（外包强度口径、主产品口径、firm-year 计数）。